# **Model Evaluation**

This notebook evaluates the performance of the resnet-18 model trained on a split of SID and CIFAKE data.
Model performance is evaluated on
- internal testing splits, and an external unseen dataset
- clean data, and augmented data for robustness

## **Mount Google Drive**

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## **Set up project**

In [2]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path("/content/ai-image-detector")

if not PROJECT_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mikkichan22/AI-image-detector.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

(PROJECT_ROOT / "src" / "__init__.py").touch()

print("Working directory:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Working directory: /content/ai-image-detector
Project exists: True
src exists: True


## **Load the trained model**

In [4]:
import os
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path("/content/ai-image-detector")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train_resnet import build_model, ManifestDataset
from src.transforms import evaluation_transform

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint_path = Path(
    "/content/drive/MyDrive/ai-image-detector/"
    "checkpoints/sid_priority_augmented/"
    "resnet18_clean_best.pth"
)

print("Checkpoint exists:", checkpoint_path.exists())

model = build_model()

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Model loaded successfully")
print("Using device:", device)

Checkpoint exists: True
Model loaded successfully
Using device: cuda


# **Model evaluation using testing split**

In [ ]:
import json

metrics_path = (
    "/content/drive/MyDrive/ai-image-detector/"
    "results/sid_priority_augmented/test_metrics.json"
)

with open(metrics_path) as file:
    metrics = json.load(file)

print(json.dumps(metrics, indent=2))

{
  "model": "resnet18",
  "training_type": "clean",
  "best_epoch": 10,
  "best_validation_f1": 0.9714173073132268,
  "test_accuracy": 0.9445181160970184,
  "test_precision": 0.9263378465506125,
  "test_recall": 0.9677527995285005,
  "test_f1": 0.9465925468396129,
  "confusion_matrix": [
    [
      10586,
      914
    ],
    [
      383,
      11494
    ]
  ],
  "classification_report": {
    "real": {
      "precision": 0.9650834169021789,
      "recall": 0.9205217391304348,
      "f1-score": 0.9422760247452046,
      "support": 11500.0
    },
    "AI/fake": {
      "precision": 0.9263378465506125,
      "recall": 0.9677527995285005,
      "f1-score": 0.9465925468396129,
      "support": 11877.0
    },
    "accuracy": 0.9445181160970184,
    "macro avg": {
      "precision": 0.9457106317263957,
      "recall": 0.9441372693294676,
      "f1-score": 0.9444342857924087,
      "support": 23377.0
    },
    "weighted avg": {
      "precision": 0.9453982075483031,
      "recall": 0.94451

## **Overall Performance**

| Metric         | Result | Meaning                                            |
| -------------- | -----: | -------------------------------------------------- |
| Test accuracy  | 94.45% | 94.45% of test images were classified correctly    |
| Test precision | 92.63% | Of images predicted as AI, 92.63% were actually AI |
| Test recall    | 96.78% | The model detected 96.78% of AI images             |
| Test F1        | 94.66% | Overall balance between precision and recall       |


**Real images:**
Correctly classified as real: 10,586
Incorrectly classified as AI: 914

Approximately 8% of real images are false positives:


**AI images:**
Correctly classified as AI: 11,494
Incorrectly classified as real: 383

Only approximately 3.2% of AI images were missed.

**What this says about the model:**

The model is optimized toward detecting AI images aggressively:

## **Perform evaluation for CIFAKE and SID testing splits seperately**
Since testing split for CIFAKE is a lot larger than SID, we test them separately to see how the model performs on the 2 datasets

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

splits = pd.read_csv(
    "data/sid_priority_splits.csv"
)

test_rows = splits[
    splits["split"] == "test"
]

for source_name in ["SID_Set", "CIFAKE"]:

    source_rows = test_rows[
        test_rows["source_dataset"] == source_name
    ].reset_index(drop=True)

    dataset = ManifestDataset(
        source_rows,
        PROJECT_ROOT,
        evaluation_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels_batch in loader:
            images = images.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels_batch.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    print(f"\n===== {source_name} TEST RESULTS =====")
    print("Images:", len(source_rows))

    print(
        "Accuracy:",
        accuracy_score(
            all_labels,
            all_predictions,
        ),
    )

    print(
        "Precision:",
        precision_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        "Recall:",
        recall_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        "F1:",
        f1_score(
            all_labels,
            all_predictions,
            zero_division=0,
        ),
    )

    print(
        classification_report(
            all_labels,
            all_predictions,
            target_names=["REAL", "AI/fake"],
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            all_labels,
            all_predictions,
            labels=[0, 1],
        )
    )


===== SID_Set TEST RESULTS =====
Images: 3000
Accuracy: 0.993
Precision: 0.9881188118811881
Recall: 0.998
F1: 0.9930348258706467
              precision    recall  f1-score   support

        REAL       1.00      0.99      0.99      1500
     AI/fake       0.99      1.00      0.99      1500

    accuracy                           0.99      3000
   macro avg       0.99      0.99      0.99      3000
weighted avg       0.99      0.99      0.99      3000

Confusion matrix:
[[1482   18]
 [   3 1497]]

===== CIFAKE TEST RESULTS =====
Images: 20378
Accuracy: 0.9384139758563156
Precision: 0.9192940527622024
Recall: 0.9636731547504336
F1: 0.9409606247353813
              precision    recall  f1-score   support

        REAL       0.96      0.91      0.94     10000
     AI/fake       0.92      0.96      0.94     10378

    accuracy                           0.94     20378
   macro avg       0.94      0.94      0.94     20378
weighted avg       0.94      0.94      0.94     20378

Confusion matri

Since testing split for CIFAKE dataset (20378 images) is much larger compared to SID testing split (3000) dataset, we evaluate the model on them separately. The difference in testing split sizes is due to hardware limitations, i did not have the storage to download the entire testing split in the SID dataset.

## **Takeaways**
From the results above, the model performs better on SID dataset compared to CIFAKE (higher F1 score of 99.3% for SID compared to 94.1% for CIFAKE). This is expected as the model is trained and validated on predominantly SID images, this was an intentional training choice as the images from SID dataset are more varied and higher resolution compared to CIFAKE dataset, making it a better representation of most actual AI/ Real images.

Moreover, the false positive rate on real CIFAKE images is quite high at 878 / 10000 = 8.78%. The others are not concern

# **Evaluate model on augmented versions of the images in the training split**

## **Create a robustness manifest**

In [ ]:
!python src/create_robustness_sets.py

## **Checking if transformation occured as intended**

In [ ]:
robustness_manifest = pd.read_csv(
    "data/robustness_manifest.csv"
)

print(robustness_manifest.head())
print(robustness_manifest["transform"].value_counts())
print("Total transformed images:", len(robustness_manifest))

                             original_path  \
0  data/raw/CIFAKE/train/FAKE/5242 (3).jpg   
1  data/raw/CIFAKE/train/FAKE/5242 (3).jpg   
2  data/raw/CIFAKE/train/FAKE/5242 (3).jpg   
3  data/raw/CIFAKE/train/FAKE/5242 (3).jpg   
4  data/raw/CIFAKE/train/FAKE/5242 (3).jpg   

                       transformed_path  label   transform parameter  
0       data/robustness/clean/00001.jpg      1       clean      none  
1     data/robustness/jpeg_70/00001.jpg      1     jpeg_70        70  
2     data/robustness/jpeg_30/00001.jpg      1     jpeg_30        30  
3    data/robustness/blur_1.0/00001.jpg      1    blur_1.0       1.0  
4  data/robustness/resize_0.5/00001.jpg      1  resize_0.5       0.5  
transform
clean            23377
jpeg_70          23377
jpeg_30          23377
blur_1.0         23377
resize_0.5       23377
noise_0.05       23377
colour_jitter    23377
crop_80          23377
Name: count, dtype: int64
Total transformed images: 187016


## **Created pytorch dataset for robustness images**
Converts images into format ResNet-18 accepts

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class RobustnessDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(
            row["transformed_path"]
        ).convert("RGB")

        image = self.transform(image)

        return (
            image,
            int(row["label"]),
            row["transform"],
            row["transformed_path"],
        )

### **Create dataset and loader**

In [ ]:
from src.transforms import evaluation_transform

robustness_dataset = RobustnessDataset(
    robustness_manifest,
    evaluation_transform,
)

robustness_loader = DataLoader(
    robustness_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("Images to evaluate:", len(robustness_dataset))

Images to evaluate: 187016


## **Run the model inference**

In [ ]:
import numpy as np
from tqdm.auto import tqdm

results = []

with torch.no_grad():
    for images, labels, transforms_used, paths in tqdm(
        robustness_loader
    ):
        images = images.to(device)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = logits.argmax(dim=1)
        ai_probabilities = probabilities[:, 1]

        for label, prediction, ai_probability, transform_name, path in zip(
            labels.numpy(),
            predictions.cpu().numpy(),
            ai_probabilities.cpu().numpy(),
            transforms_used,
            paths,
        ):
            results.append({
                "image_path": path,
                "label": int(label),
                "prediction": int(prediction),
                "ai_probability": float(ai_probability),
                "transform": transform_name,
            })

predictions_df = pd.DataFrame(results)

print(predictions_df.head())
print("Predictions:", len(predictions_df))

  0%|          | 0/1462 [00:00<?, ?it/s]

                             image_path  label  prediction  ai_probability  \
0       data/robustness/clean/00001.jpg      1           0        0.019555   
1     data/robustness/jpeg_70/00001.jpg      1           0        0.023334   
2     data/robustness/jpeg_30/00001.jpg      1           0        0.019295   
3    data/robustness/blur_1.0/00001.jpg      1           0        0.001892   
4  data/robustness/resize_0.5/00001.jpg      1           0        0.001591   

    transform  
0       clean  
1     jpeg_70  
2     jpeg_30  
3    blur_1.0  
4  resize_0.5  
Predictions: 187016


### save the predictions to a csv file

In [ ]:
predictions_path = Path(
    "results/robustness_predictions.csv"
)

predictions_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

print("Saved to:", predictions_path)

Saved to: results/robustness_predictions.csv


## **Calculate metrics for every transformation**

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

metrics = []

for transform_name, group in predictions_df.groupby("transform"):
    labels = group["label"]
    predictions = group["prediction"]

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    metrics.append({
        "transform": transform_name,
        "images": len(group),
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "false_positive_rate": fp / (fp + tn),
        "false_negative_rate": fn / (fn + tp),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    })

metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df.sort_values("transform")

display(metrics_df)

,transform,images,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives
0,blur_1.0,23377,0.747872,0.966474,0.521849,0.677747,0.018696,0.478151,11285,215,5679,6198
1,clean,23377,0.946614,0.932883,0.964301,0.948332,0.071652,0.035699,10676,824,424,11453
2,colour_jitter,23377,0.905548,0.858722,0.974404,0.912913,0.165565,0.025596,9596,1904,304,11573
3,crop_80,23377,0.799718,0.988459,0.612949,0.756678,0.007391,0.387051,11415,85,4597,7280
4,jpeg_30,23377,0.911409,0.945727,0.875895,0.909472,0.051913,0.124105,10903,597,1474,10403
5,jpeg_70,23377,0.940240,0.918664,0.968090,0.942729,0.088522,0.031910,10482,1018,379,11498
6,noise_0.05,23377,0.862215,0.942989,0.775701,0.851203,0.048435,0.224299,10943,557,2664,9213
7,resize_0.5,23377,0.738247,0.973364,0.498442,0.659279,0.014087,0.501558,11338,162,5957,5920


## **Analysing results**

1. Clean performance is good
On clean data:
- Accuracy: 94.66%
- F1: 94.83%
- AI recall: 96.43%

The model detects AI images effectively, but falsely flags around 7.17% of real images.

2. JPEG quality 70 produces almost the same result as clean data:
- Clean F1:   94.83%
- JPEG 70 F1: 94.27%

The model is reasonably tolerant of moderate compression.

3. Severe compression causes missed AI images

- JPEG quality 30 reduces AI recall to 87.59%.
- The model misses: 1,474 AI images

This suggests some of the model’s detection signal is removed by strong compression.

4. Model is weak to blur and resizing

Blur is the most damaging transformation:
- Accuracy: 74.79%
- AI recall: 52.18%
The model misses almost half of the AI images.

Resize 50% is similarly bad:
- Accuracy: 73.82%
- AI recall: 49.84%

This strongly suggests the model depends on high-frequency texture or small pixel-level artifacts. Once those details are blurred or downsampled, the model often predicts REAL.

5. Cropping also causes many false negatives

With the 80% crop:
- AI recall: 61.29%
- False-negative rate: 38.71%

6. Colour jitter creates false accusations

Colour jitter gives:
- AI recall: 97.44%
- False-positive rate: 16.56%

The model still detects AI images, but it incorrectly labels many real images as AI. This means the model is sensitive to colour statistics. Colour changes move real images into a feature space that resembles the AI class.

**Overall diagnosis**
- Good on clean images
- Good under moderate JPEG compression
- Weak under blur and downsampling
- Sensitive to colour changes

# **Evaluating the model on an external dataset**
To further test the model, we're evaluating it on a small external dataset to see how well it performs on data that it has not seen in training/ validation.

In line with the problem statement, we'll be using the WildFake dataset, more specifically:
- AIGC: DALL·E Advanced
- Non-AIGC: COCO val2017

However due to space and hardware limitations, we cannot test on the entire subset stated in the problem statement. Hence, we'll be taking a small balanced subset of the above images, with 300 AIGC and 300 non-AIGC


In [5]:
!pip install -q datasets pillow

## **Set output directory**

In [6]:
from pathlib import Path

PROJECT_ROOT = Path("/content/ai-image-detector")
EVAL_ROOT = PROJECT_ROOT / "eval_data"

REAL_DIR = EVAL_ROOT / "real"
AI_DIR = EVAL_ROOT / "ai"

REAL_DIR.mkdir(parents=True, exist_ok=True)
AI_DIR.mkdir(parents=True, exist_ok=True)

print("Saving images to:", EVAL_ROOT)

Saving images to: /content/ai-image-detector/eval_data


## **Load WildFake dataset**

In [7]:
from datasets import load_dataset

wildfake_stream = load_dataset(
    "techjam-aigc/wildfake-eval-subset",
    split="validation",
    streaming=True,
)

README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

## **Save a small balanced subset of the WildFake dataset**

In [9]:
import random
import pandas as pd
from collections import Counter

TARGET_PER_CLASS = 300
random.seed(42)

wildfake_stream = load_dataset(
    "techjam-aigc/wildfake-eval-subset",
    split="validation",
    streaming=True,
)

# Shuffle the stream approximately without storing the whole dataset.
wildfake_stream = wildfake_stream.shuffle(
    seed=42,
    buffer_size=1000,
)

counts = Counter()
records = []

for row in wildfake_stream:

    label = int(row["label"])
    source = row["source"]

    # Keep only the official challenge sources.
    if source == "coco_val2017" and label == 0:
        output_dir = REAL_DIR

    elif source == "dalle3_advanced" and label == 1:
        output_dir = AI_DIR

    else:
        continue

    if counts[label] >= TARGET_PER_CLASS:
        continue

    image = row["image"].convert("RGB")

    image_number = counts[label]

    if label == 0:
        output_path = output_dir / f"real_{image_number:04d}.png"
    else:
        output_path = output_dir / f"ai_{image_number:04d}.png"

    # PNG avoids adding another lossy JPEG compression step.
    image.save(output_path, format="PNG")

    records.append({
        "image_path": str(output_path),
        "label": label,
        "class_name": "REAL" if label == 0 else "AI/fake",
        "source_dataset": source,
        "original_id": row["id"],
        "split": "external_test",
    })

    counts[label] += 1

    if (
        counts[0] >= TARGET_PER_CLASS
        and counts[1] >= TARGET_PER_CLASS
    ):
        break

print("Saved counts:", dict(counts))

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/techjam-aigc/wildfake-eval-subset/resolve/22b1b7ae288778f12dbbb0db70fbe5d87b71b873/data/validation-00004-of-00008.parquet
Retrying in 1s [Retry 1/5].


Saved counts: {1: 300, 0: 300}


In [10]:
!find /content/ai-image-detector/eval_data/real -type f | wc -l
!find /content/ai-image-detector/eval_data/ai -type f | wc -l
!du -sh /content/ai-image-detector/eval_data

300
300
434M	/content/ai-image-detector/eval_data


## **Save the evaluation manifest**

In [11]:
manifest = pd.DataFrame(records)

manifest_path = PROJECT_ROOT / "data" / "external_eval.csv"
manifest.to_csv(manifest_path, index=False)

print("Created:", manifest_path)
display(manifest.groupby(["source_dataset", "label"]).size())

Created: /content/ai-image-detector/data/external_eval.csv


,,0
source_dataset,label,
coco_val2017,0,300
dalle3_advanced,1,300


## **save new images into a dataframe**

In [20]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/ai-image-detector")
EVAL_ROOT = PROJECT_ROOT / "eval_data"

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

records = []

# Real images: label 0
for image_path in sorted(
    (EVAL_ROOT / "real").rglob("*")
):
    if image_path.suffix.lower() in valid_extensions:
        records.append({
            "image_path": str(image_path),
            "label": 0,
            "class_name": "REAL",
        })

# AI images: label 1
for image_path in sorted(
    (EVAL_ROOT / "ai").rglob("*")
):
    if image_path.suffix.lower() in valid_extensions:
        records.append({
            "image_path": str(image_path),
            "label": 1,
            "class_name": "AI/fake",
        })

external_df = pd.DataFrame(records)

print("Total images:", len(external_df))
print(external_df.groupby(["class_name", "label"]).size())

Total images: 600
class_name  label
AI/fake     1        300
REAL        0        300
dtype: int64


## **Create a dataset for this**

In [12]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class ExternalImageDataset(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        valid_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".webp",
        }

        # Real images have label 0.
        for image_path in sorted(
            (self.root_dir / "real").rglob("*")
        ):
            if image_path.suffix.lower() in valid_extensions:
                self.samples.append(
                    (image_path, 0)
                )

        # AI images have label 1.
        for image_path in sorted(
            (self.root_dir / "ai").rglob("*")
        ):
            if image_path.suffix.lower() in valid_extensions:
                self.samples.append(
                    (image_path, 1)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        image = self.transform(image)

        return image, label, str(image_path)

In [13]:
from src.transforms import evaluation_transform

eval_dataset = ExternalImageDataset(
    EVAL_ROOT,
    evaluation_transform,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print("Total evaluation images:", len(eval_dataset))

Total evaluation images: 600


## **Generate predictions**

In [14]:
from tqdm.auto import tqdm

all_labels = []
all_predictions = []
all_ai_probabilities = []
all_paths = []

model.eval()

with torch.inference_mode():

    for images, labels, paths in tqdm(
        eval_loader,
        desc="Evaluating external dataset",
    ):
        images = images.to(device)

        outputs = model(images)
        probabilities = torch.softmax(
            outputs,
            dim=1,
        )

        predictions = outputs.argmax(dim=1)
        ai_probabilities = probabilities[:, 1]

        all_labels.extend(
            labels.tolist()
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_ai_probabilities.extend(
            ai_probabilities.cpu().tolist()
        )

        all_paths.extend(paths)

print(
    "Finished evaluating:",
    len(all_predictions),
    "images",
)

Evaluating external dataset:   0%|          | 0/19 [00:00<?, ?it/s]

Finished evaluating: 600 images


## **Calculate Metrics**

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

accuracy = accuracy_score(
    all_labels,
    all_predictions,
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["REAL", "AI/fake"],
        zero_division=0,
    )
)

matrix = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1],
)

print("Confusion matrix:")
print(matrix)

Accuracy: 0.8
Precision: 0.96875
Recall: 0.62
F1: 0.7560975609756098
              precision    recall  f1-score   support

        REAL       0.72      0.98      0.83       300
     AI/fake       0.97      0.62      0.76       300

    accuracy                           0.80       600
   macro avg       0.84      0.80      0.79       600
weighted avg       0.84      0.80      0.79       600

Confusion matrix:
[[294   6]
 [114 186]]


## **Analysing results**

On a held-out external dataset of 600 images, the model achieved 80.0% accuracy and an AI-detection F1 score of 75.6%, which is lower than the F1 score when tested on CIFAKE and SID datasets. Unfortunately, the model performs well on the datasets it knows, but struggles with unfamiliar images.

It achieved 96.9% precision but only 62.0% recall for AI-generated images, indicating that it was conservative and missed many synthetic images. The 2.0% false-positive rate shows strong protection against incorrectly flagging real images, but the 38.0% false-negative rate indicates limited generalisation to unseen data.

## **Testing on augmented external dataset**
Applying augmentation transformations to the above external dataset and testing the model on it

Robustness transformations:
- JPEG quality 90, 70, 50, 30
- Blur sigma 0.5, 1.0, 2.0
- Resize 0.5 and 0.25, then upscale
- Gaussian noise sigma 0.02, 0.05, 0.10
- Color adjustment ±20%
- Center crop 80%

## **Define the transformations**

In [17]:
import io
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

def apply_robustness_transform(
    image,
    transform_name,
    image_index=0,
):
    image = image.convert("RGB")

    # No transformation.
    if transform_name == "clean":
        return image

    # JPEG compression.
    if transform_name.startswith("jpeg_"):
        quality = int(
            transform_name.replace("jpeg_", "")
        )

        buffer = io.BytesIO()
        image.save(
            buffer,
            format="JPEG",
            quality=quality,
        )

        buffer.seek(0)

        with Image.open(buffer) as compressed:
            return compressed.convert("RGB")

    # Gaussian blur.
    if transform_name.startswith("blur_"):
        sigma = float(
            transform_name.replace("blur_", "")
        )

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=sigma
            )
        )

    # Resize down, then resize back up.
    if transform_name.startswith("resize_"):
        scale = float(
            transform_name.replace("resize_", "")
        )

        width, height = image.size

        small_width = max(1, int(width * scale))
        small_height = max(1, int(height * scale))

        smaller = image.resize(
            (small_width, small_height),
            Image.Resampling.BILINEAR,
        )

        return smaller.resize(
            (width, height),
            Image.Resampling.BILINEAR,
        )

    # Gaussian noise.
    if transform_name.startswith("noise_"):
        sigma = float(
            transform_name.replace("noise_", "")
        )

        array = (
            np.asarray(image)
            .astype(np.float32)
            / 255.0
        )

        # Makes the noise reproducible for each image.
        rng = np.random.default_rng(
            42 + image_index
        )

        noise = rng.normal(
            loc=0.0,
            scale=sigma,
            size=array.shape,
        )

        noisy = np.clip(
            array + noise,
            0.0,
            1.0,
        )

        return Image.fromarray(
            (noisy * 255).astype(np.uint8)
        )

    # Fixed +20% brightness, contrast, and saturation.
    if transform_name == "colour_jitter":
        image = ImageEnhance.Brightness(
            image
        ).enhance(1.2)

        image = ImageEnhance.Contrast(
            image
        ).enhance(1.2)

        image = ImageEnhance.Color(
            image
        ).enhance(1.2)

        return image

    # Center crop 80%, then resize to original dimensions.
    if transform_name == "crop_80":
        width, height = image.size

        crop_width = int(width * 0.8)
        crop_height = int(height * 0.8)

        left = (width - crop_width) // 2
        top = (height - crop_height) // 2

        cropped = image.crop((
            left,
            top,
            left + crop_width,
            top + crop_height,
        ))

        return cropped.resize(
            (width, height),
            Image.Resampling.BILINEAR,
        )

    raise ValueError(
        f"Unknown transformation: {transform_name}"
    )

In [18]:
TRANSFORM_NAMES = [
    "clean",
    "jpeg_30",
    "jpeg_70",
    "blur_1.0",
    "resize_0.5",
    "noise_0.05",
    "colour_jitter",
    "crop_80",
]

print("Number of conditions:", len(TRANSFORM_NAMES))

Number of conditions: 8


In [21]:
evaluation_df = external_df.copy()

## **Create the transformed dataset**

In [22]:
from PIL import Image
from torch.utils.data import Dataset

class RobustnessDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform_name,
        model_transform,
        project_root,
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.transform_name = transform_name
        self.model_transform = model_transform
        self.project_root = Path(project_root)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = Path(row["image_path"])

        if not image_path.is_absolute():
            image_path = self.project_root / image_path

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        image = apply_robustness_transform(
            image=image,
            transform_name=self.transform_name,
            image_index=index,
        )

        # Resize and normalize exactly as during evaluation.
        image = self.model_transform(image)

        label = int(row["label"])

        return image, label, str(image_path)

## **Generate predictions and calculate metrics**

In [23]:
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

def evaluate_one_condition(
    dataframe,
    transform_name,
):
    dataset = RobustnessDataset(
        dataframe=dataframe,
        transform_name=transform_name,
        model_transform=evaluation_transform,
        project_root=PROJECT_ROOT,
    )

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )

    labels = []
    predictions = []
    ai_probabilities = []
    paths = []

    model.eval()

    with torch.inference_mode():

        for images, batch_labels, batch_paths in tqdm(
            loader,
            desc=f"Evaluating {transform_name}",
        ):
            images = images.to(device)

            outputs = model(images)

            probabilities = torch.softmax(
                outputs,
                dim=1,
            )

            batch_predictions = outputs.argmax(
                dim=1
            )

            labels.extend(
                batch_labels.tolist()
            )

            predictions.extend(
                batch_predictions.cpu().tolist()
            )

            ai_probabilities.extend(
                probabilities[:, 1].cpu().tolist()
            )

            paths.extend(batch_paths)

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    )

    tn, fp, fn, tp = matrix.ravel()

    result = {
        "transform": transform_name,
        "images": len(labels),
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "false_positive_rate": fp / (fp + tn),
        "false_negative_rate": fn / (fn + tp),
    }

    predictions_df = pd.DataFrame({
        "image_path": paths,
        "label": labels,
        "prediction": predictions,
        "ai_probability": ai_probabilities,
        "transform": transform_name,
    })

    return result, predictions_df

In [24]:
import json

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/ai-image-detector/"
    "results/external_robustness"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

metrics_results = []
prediction_results = []

for transform_name in TRANSFORM_NAMES:

    result, predictions_df = evaluate_one_condition(
        dataframe=evaluation_df,
        transform_name=transform_name,
    )

    metrics_results.append(result)
    prediction_results.append(predictions_df)

    current_metrics = pd.DataFrame(
        metrics_results
    )

    current_predictions = pd.concat(
        prediction_results,
        ignore_index=True,
    )

    current_metrics.to_csv(
        OUTPUT_DIR / "metrics.csv",
        index=False,
    )

    current_predictions.to_csv(
        OUTPUT_DIR / "predictions.csv",
        index=False,
    )

    print(
        f"{transform_name}: "
        f"F1={result['f1']:.4f}, "
        f"Accuracy={result['accuracy']:.4f}"
    )

print("Finished all transformations.")
print("Results saved to:", OUTPUT_DIR)

Evaluating clean:   0%|          | 0/19 [00:00<?, ?it/s]

clean: F1=0.7561, Accuracy=0.8000


Evaluating jpeg_30:   0%|          | 0/19 [00:00<?, ?it/s]

jpeg_30: F1=0.7303, Accuracy=0.7833


Evaluating jpeg_70:   0%|          | 0/19 [00:00<?, ?it/s]

...

noise_0.05: F1=0.6940, Accuracy=0.7633


Evaluating colour_jitter:   0%|          | 0/19 [00:00<?, ?it/s]

colour_jitter: F1=0.7480, Accuracy=0.7933


Evaluating crop_80:   0%|          | 0/19 [00:00<?, ?it/s]

crop_80: F1=0.6925, Accuracy=0.7617
Finished all transformations.
Results saved to: /content/drive/MyDrive/ai-image-detector/results/external_robustness


In [27]:
metrics_df = pd.DataFrame(metrics_results)
metrics_df = metrics_df.sort_values("transform")
display(metrics_df)

,transform,images,accuracy,precision,recall,f1,true_negatives,false_positives,false_negatives,true_positives,false_positive_rate,false_negative_rate
3,blur_1.0,600,0.786667,0.967391,0.593333,0.735537,294,6,122,178,0.020000,0.406667
0,clean,600,0.800000,0.968750,0.620000,0.756098,294,6,114,186,0.020000,0.380000
6,colour_jitter,600,0.793333,0.958333,0.613333,0.747967,292,8,116,184,0.026667,0.386667
7,crop_80,600,0.761667,0.975758,0.536667,0.692473,296,4,139,161,0.013333,0.463333
1,jpeg_30,600,0.783333,0.967033,0.586667,0.730290,294,6,124,176,0.020000,0.413333
2,jpeg_70,600,0.786667,0.957447,0.600000,0.737705,292,8,120,180,0.026667,0.400000
5,noise_0.05,600,0.763333,0.981707,0.536667,0.693966,297,3,139,161,0.010000,0.463333
4,resize_0.5,600,0.758333,0.905759,0.576667,0.704684,282,18,127,173,0.060000,0.423333


## **Analysing results**

F1 score doesn't change too drastically between the different transformations.
The model was relatively stable under blur, JPEG compression, and colour adjustment, but was more sensitive to cropping, resizing, and Gaussian noise. The main degradation occurred through reduced AI recall rather than increased false positives.